# 1) Import Data
* Import Dependencies
* Import CSV

### Import Dependencies

In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
from mapping import MILITARY_BASES

### Import CSV

In [2]:
data_raw = pd.read_csv("aircraft_data_new.csv")

# Convert 'timestamp' column to datetime
data_raw["datetime"] = pd.to_datetime(data_raw["datetime"])

# Sort by hex and then datetime
data_raw = data_raw.sort_values(by=["hex", "datetime"]).reset_index(drop=True)

print("Unique hex codes: ", data_raw['hex'].nunique())
print("Unique callsigns: ", data_raw['callsign'].nunique())
data_raw.head(10)

Unique hex codes:  12
Unique callsigns:  12


,hex,callsign,datetime,operator,tail number,squawk,altitude,latitude,longitude,type,heading,ground speed
0,ae07f8,TEDDY45,2025-05-25 13:45:24.738465,Usaf | 437aw [kchs],97-0046,1501,6150,41.664431,-73.540140,C-17A Globemaster,331.0,247.0
1,ae07f8,TEDDY45,2025-05-25 13:46:42.116406,Usaf | 437aw [kchs],97-0046,1501,5850,41.744182,-73.587190,C-17A Globemaster,334.0,232.0
2,ae07f8,TEDDY45,2025-05-25 13:47:42.872921,Usaf | 437aw [kchs],97-0046,2645,5150,41.761644,-73.651096,C-17A Globemaster,242.0,216.0
3,ae07f8,TEDDY45,2025-05-25 13:48:43.475530,Usaf | 437aw [kchs],97-0046,2645,4750,41.699014,-73.715586,C-17A Globemaster,242.0,289.0
4,ae07f8,TEDDY45,2025-05-25 13:49:44.114103,Usaf | 437aw [kchs],97-0046,2645,3875,41.658669,-73.784490,C-17A Globemaster,230.0,223.0
5,ae07f8,TEDDY45,2025-05-25 13:50:44.483272,Usaf | 437aw [kchs],97-0046,2645,3875,41.595026,-73.827341,C-17A Globemaster,190.0,272.0
6,ae07f8,TEDDY45,2025-05-25 13:51:45.148663,Usaf | 437aw [kchs],97-0046,2645,3125,41.555510,-73.851350,C-17A Globemaster,205.0,253.0
7,ae07f8,TEDDY45,2025-05-25 13:52:46.009656,Usaf | 437aw [kchs],97-0046,2645,2375,41.543893,-73.899800,C-17A Globemaster,276.0,202.0
8,ae07f8,TEDDY45,2025-05-25 13:54:47.022342,Usaf | 437aw [kchs],97-0046,2645,1525,41.557054,-73.988529,C-17A Globemaster,213.0,215.0
9,ae080e,RCH808,2025-05-25 13:45:24.738456,United States Air Force,99-0168,0750,20850,35.141797,-79.702955,C-17A Globemaster,27.0,432.0


# 2) Transform Data
* Base Metrics

### Base Metrics

In [3]:
max_distance_miles = 15

def find_nearest_base(row):
  """
  Checks if a given lat/lon is within max_distance_miles of any base.
  If so, keep track of the base that is the closet and add that info.
  """
  benchmark = max_distance_miles
  row['near base'] = None
  row['distance to base'] = None
  point = (row['latitude'], row['longitude'])
  if row['altitude'] <= 10000:
    for base_name, coords in MILITARY_BASES.items():
      base_point = (coords['lat'], coords['lon'])
      distance = geodesic(point, base_point).miles
      if distance <= benchmark:
        # This distance becomes the benchmark
        benchmark = distance
        row['near base'] = base_name
        row['distance to base'] = int(distance)
  return row

# --- Apply the function to your DataFrame ---
df = data_raw.apply(lambda row: find_nearest_base(row), axis=1)

df.head()

,hex,callsign,datetime,operator,tail number,squawk,altitude,latitude,longitude,type,heading,ground speed,near base,distance to base
0,ae07f8,TEDDY45,2025-05-25 13:45:24.738465,Usaf | 437aw [kchs],97-0046,1501,6150,41.664431,-73.540140,C-17A Globemaster,331.0,247.0,None,NaN
1,ae07f8,TEDDY45,2025-05-25 13:46:42.116406,Usaf | 437aw [kchs],97-0046,1501,5850,41.744182,-73.587190,C-17A Globemaster,334.0,232.0,None,NaN
2,ae07f8,TEDDY45,2025-05-25 13:47:42.872921,Usaf | 437aw [kchs],97-0046,2645,5150,41.761644,-73.651096,C-17A Globemaster,242.0,216.0,None,NaN
3,ae07f8,TEDDY45,2025-05-25 13:48:43.475530,Usaf | 437aw [kchs],97-0046,2645,4750,41.699014,-73.715586,C-17A Globemaster,242.0,289.0,None,NaN
4,ae07f8,TEDDY45,2025-05-25 13:49:44.114103,Usaf | 437aw [kchs],97-0046,2645,3875,41.658669,-73.784490,C-17A Globemaster,230.0,223.0,None,NaN


In [4]:
test = df[~df['near base'].isna()]

test.head(50)

,hex,callsign,datetime,operator,tail number,squawk,altitude,latitude,longitude,type,heading,ground speed,near base,distance to base
6,ae07f8,TEDDY45,2025-05-25 13:51:45.148663,Usaf | 437aw [kchs],97-0046,2645,3125,41.555510,-73.851350,C-17A Globemaster,205.0,253.0,Stewart ANGB,13.0
7,ae07f8,TEDDY45,2025-05-25 13:52:46.009656,Usaf | 437aw [kchs],97-0046,2645,2375,41.543893,-73.899800,C-17A Globemaster,276.0,202.0,Stewart ANGB,10.0
8,ae07f8,TEDDY45,2025-05-25 13:54:47.022342,Usaf | 437aw [kchs],97-0046,2645,1525,41.557054,-73.988529,C-17A Globemaster,213.0,215.0,Stewart ANGB,6.0
282,ae117d,RCH406,2025-05-25 16:18:18.642410,United States Air Force,02-1111,6115,2875,27.865225,-82.741426,C-17A Globemaster,175.0,272.0,MacDill AFB,13.0
283,ae117d,RCH406,2025-05-25 16:19:19.285434,United States Air Force,02-1111,6115,2200,27.798128,-82.733008,C-17A Globemaster,172.0,260.0,MacDill AFB,13.0
284,ae117d,RCH406,2025-05-25 16:20:19.640257,United States Air Force,02-1111,6115,1475,27.776555,-82.657906,C-17A Globemaster,74.0,307.0,MacDill AFB,9.0
285,ae117d,RCH406,2025-05-25 16:21:20.055590,United States Air Force,02-1111,6115,1475,27.793981,-82.572975,C-17A Globemaster,56.0,273.0,MacDill AFB,4.0
286,ae117d,RCH406,2025-05-25 16:22:20.876624,United States Air Force,02-1111,6115,1500,27.850454,-82.519073,C-17A Globemaster,41.0,250.0,MacDill AFB,0.0
287,ae117d,RCH406,2025-05-25 16:23:21.532647,United States Air Force,02-1111,6115,1500,27.821163,-82.499511,C-17A Globemaster,227.0,215.0,MacDill AFB,2.0
288,ae117d,RCH406,2025-05-25 16:24:21.917337,United States Air Force,02-1111,6115,1500,27.785597,-82.531955,C-17A Globemaster,213.0,170.0,MacDill AFB,4.0
